# Bula Clara — Treino da LLM (LoRA) no Google Colab (GPU T4)

**Issue #2:** gerar o adaptador LoRA do grupo (`Qwen/Qwen2.5-1.5B-Instruct`), salvar `historico_treino.json` e gerar o gráfico da curva de perda.

### Antes de começar
1. Ative a GPU: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**.
2. Envie seu `data/chunks.jsonl` para `/content/chunks.jsonl` (ícone de pasta à esquerda) **ou** coloque-o no Google Drive e ligue `USAR_DRIVE = True` na célula 3.
3. **Não commite os pesos do adaptador no GitHub** — salve no Google Drive ou em uma Release do repositório.

In [ ]:
BRANCH = "dev"  # ajuste se for rodar a partir de outra branch

!rm -rf challenge_2_bula-clara
!git clone --branch {BRANCH} https://github.com/bula-clara-unb/challenge_2_bula-clara.git
%cd challenge_2_bula-clara
!pip -q install -U transformers peft accelerate scikit-learn pypdf chromadb sentence-transformers fastapi matplotlib

import torch
assert torch.cuda.is_available(), "GPU não detectada! Ative T4 em Ambiente de execução -> Alterar tipo."
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import json
import shutil
from pathlib import Path

USAR_DRIVE = False
CAMINHO_CHUNKS_DRIVE = "/content/drive/MyDrive/bula-clara/chunks.jsonl"
PASTA_SAIDA_DRIVE = "/content/drive/MyDrive/bula-clara/entrega_issue_2"

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    origem = Path(CAMINHO_CHUNKS_DRIVE)
else:
    origem = Path("/content/chunks.jsonl")

assert origem.exists(), f"Arquivo não encontrado em {origem}. Envie data/chunks.jsonl antes de continuar."
Path("data").mkdir(exist_ok=True)
shutil.copy(origem, "data/chunks.jsonl")

chunks = [json.loads(l) for l in Path("data/chunks.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
meds = sorted({c["medicamento"] for c in chunks})
print(f"{len(chunks)} chunks carregados de {len(meds)} medicamentos: {', '.join(meds)}")

In [ ]:
!python scripts/gerar_dataset_finetune.py

from pathlib import Path
import json
print("\nResumo do dataset:")
print(Path("data/finetune/resumo_dataset.json").read_text(encoding="utf-8"))
primeiro = json.loads(Path("data/finetune/treino.jsonl").read_text(encoding="utf-8").splitlines()[0])
print("\nExemplo de resposta-alvo:", primeiro["messages"][-1]["content"])

In [ ]:
!python scripts/treinar_lora.py --epocas 3 --rank 16 --lr 2e-4

In [ ]:
from pathlib import Path
from IPython.display import Image, display

display(Image(filename="resultados/grafico_perda_lora.png"))
print(Path("resultados/relatorio_treino_lora.md").read_text(encoding="utf-8"))

In [ ]:
# Avaliação comparativa na GPU: modelo base vs LoRA com ChromaDB
!python -m bula_clara.recuperador
!BULA_LLM=hf BULA_RECUPERADOR=chroma python scripts/avaliar.py --sem-lora --rotulo base
!BULA_LLM=hf BULA_RECUPERADOR=chroma python scripts/avaliar.py --modos rag multiagente --rotulo lora

In [ ]:
# Salva adaptador, historico_treino.json, gráfico e resultados (SEM commitar pesos no Git)
!zip -qr lora_e_resultados.zip modelos/bula-clara-lora resultados data/finetune/resumo_dataset.json

if USAR_DRIVE:
    import shutil
    from pathlib import Path
    destino_drive = Path(PASTA_SAIDA_DRIVE)
    destino_drive.mkdir(parents=True, exist_ok=True)
    shutil.copy("lora_e_resultados.zip", destino_drive / "lora_e_resultados.zip")
    shutil.copytree("modelos/bula-clara-lora", destino_drive / "bula-clara-lora", dirs_exist_ok=True)
    shutil.copytree("resultados", destino_drive / "resultados", dirs_exist_ok=True)
    print(f"Arquivos salvos no Drive em {destino_drive}")
else:
    from google.colab import files
    files.download("lora_e_resultados.zip")